# LeetCode 2932 — Maximum Strong Pair XOR I bằng Python

Link đề: https://leetcode.com/problems/maximum-strong-pair-xor-i/ · Độ khó: **Easy** · Chủ đề: Array, Bit Manipulation, Sorting, Two Pointers, Trie

Skeleton LeetCode:

```python
from typing import List

class Solution:
    def maximumStrongPairXor(self, nums: List[int]) -> int:
        ...
```

Ghi chú: đề và các ví dụ được viết theo trí nhớ về đề gốc; mọi đáp án trong notebook đã được kiểm chứng bằng code, nhưng nên đối chiếu lại với đề gốc nếu muốn khớp từng chữ.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: tìm **cặp mạnh (strong pair)** `(x, y)` thỏa `|x - y| <= min(x, y)` có **XOR lớn nhất**; được chọn **cùng một số hai lần** (cặp `(x, x)`).
2. Hiểu **phép XOR** (`^`) và cách đọc XOR ở dạng nhị phân; biến điều kiện cặp mạnh thành **`y <= 2x`** (khi `x <= y`).
3. Đi từ cách **thử mọi cặp `O(n²)`** → **bỏ trùng và chỉ duyệt `y` trong `[x, 2x]`** → **sắp xếp + cửa sổ trượt + Trie nhị phân đếm số lượng** (ý tưởng cho Bài II).
4. Hiểu **Trie nhị phân** để tìm số có XOR lớn nhất với một số cho trước bằng cách đi **ngược bit** từ cao xuống thấp.
5. Tránh các lỗi kinh điển: quên cặp `(x, x)` (nên XOR tối thiểu là `0`), dùng `|x - y| < min` thay vì `<=`, quên rằng điều kiện `y <= 2x` chỉ đúng khi `x <= y`, và nhầm `^` (XOR) với lũy thừa.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums` (đánh chỉ số từ `0`). Một cặp số `(x, y)` gọi là **cặp mạnh (strong pair)** nếu thỏa:

`|x - y| <= min(x, y)`.

Hãy chọn **hai số nguyên** từ `nums` tạo thành một cặp mạnh sao cho **XOR** `x ^ y` **lớn nhất** và trả về giá trị XOR lớn nhất đó. **Có thể chọn cùng một số hai lần** (cặp `(x, x)`).

**Ví dụ 1**

- Input: `nums = [1, 2, 3, 4, 5]`
- Output: `7`
- Giải thích: có 11 cặp mạnh: `(1,1), (1,2), (2,2), (2,3), (2,4), (3,3), (3,4), (3,5), (4,4), (4,5), (5,5)`. XOR lớn nhất là `3 ^ 4 = 7`.

**Ví dụ 2**

- Input: `nums = [10, 100]`
- Output: `0`
- Giải thích: chỉ có hai cặp mạnh `(10, 10)` và `(100, 100)` (vì `|10 - 100| = 90 > 10`), đều có XOR `0`.

**Ví dụ 3**

- Input: `nums = [5, 6, 25, 30]`
- Output: `7`
- Giải thích: các cặp mạnh gồm `(5,5), (5,6), (6,6), (25,25), (25,30), (30,30)`. XOR lớn nhất là `25 ^ 30 = 7`.

**Ràng buộc:** `1 <= nums.length <= 50`; `1 <= nums[i] <= 100`.

In [1]:
from typing import List
import random
from bisect import bisect_left

examples = [
    ([1, 2, 3, 4, 5], 7),
    ([10, 100], 0),
    ([5, 6, 25, 30], 7),
]
for nums, expected in examples:
    print("nums =", nums, "-> kỳ vọng", expected)

nums = [1, 2, 3, 4, 5] -> kỳ vọng 7
nums = [10, 100] -> kỳ vọng 0
nums = [5, 6, 25, 30] -> kỳ vọng 7


## 3. Kiến thức nền

### 3.1. Phép XOR (`^`)

**XOR** (exclusive or) so sánh **từng bit**: kết quả bằng `1` khi hai bit **khác nhau**, bằng `0` khi **giống nhau**. Ví dụ `3 ^ 4`: `011 ^ 100 = 111 = 7`. Tính chất: `x ^ x = 0`; `x ^ 0 = x`; giao hoán. Kết quả XOR **lớn** khi hai số **khác nhau ở bit cao** nhất có thể.

In [2]:
print(bin(3), bin(4), bin(3 ^ 4), 3 ^ 4)          # 0b11 0b100 0b111 7
print(25 ^ 30, bin(25), bin(30), bin(25 ^ 30))    # 7
print(5 ^ 5, 9 ^ 0)                                # 0 9

0b11 0b100 0b111 7
7 0b11001 0b11110 0b111
0 9


### 3.2. Điều kiện cặp mạnh `|x - y| <= min(x, y)`

Điều kiện nhìn rối vì có trị tuyệt đối và `min`. Ta đơn giản hóa bằng cách **giả sử `x <= y`** (hoán đổi nếu cần):

- `|x - y| = y - x` và `min(x, y) = x`;
- điều kiện thành `y - x <= x`, tức **`y <= 2x`**.

Nghĩa là **số lớn không quá gấp đôi số nhỏ**. Ví dụ `(3, 5)`: `5 <= 6` mạnh; `(10, 100)`: `100 > 20` không mạnh.

In [3]:
def is_strong(x, y):
    return abs(x - y) <= min(x, y)


def is_strong_simple(x, y):
    lo, hi = min(x, y), max(x, y)
    return hi <= 2 * lo


pairs = [(3, 5), (10, 100), (5, 5), (25, 30), (2, 5), (4, 8), (4, 9)]
for x, y in pairs:
    print((x, y), is_strong(x, y), is_strong_simple(x, y))
    assert is_strong(x, y) == is_strong_simple(x, y)

(3, 5) True True
(10, 100) False False
(5, 5) True True
(25, 30) True True
(2, 5) False False
(4, 8) True True
(4, 9) False False


### 3.3. Cặp `(x, x)` luôn mạnh

Với `y = x`: `|x - x| = 0 <= x` (vì `x >= 1`). Nên **mọi số** tạo cặp mạnh với chính nó và XOR `x ^ x = 0`. Hệ quả: **đáp án luôn `>= 0`**, và nếu không có cặp nào khác thì đáp án là `0` (như ví dụ 2).

In [4]:
for x in [1, 7, 100]:
    print(x, is_strong(x, x), x ^ x)

1 True 0
7 True 0
100 True 0


### 3.4. `set` để bỏ số trùng, và `sorted`

Cặp mạnh chỉ phụ thuộc **giá trị**, nên số trùng không thêm thông tin: ta có thể lọc bằng `set(nums)`. `sorted(...)` sắp xếp tăng dần; sau khi sắp xếp, điều kiện `y <= 2x` cho thấy các `y` hợp lệ với một `x` là các phần tử **liên tục** trong khoảng `[x, 2x]`.

In [5]:
nums = [5, 6, 25, 30, 6, 5]
vals = sorted(set(nums))
print(vals)                                  # [5, 6, 25, 30]
x = 5
print([y for y in vals if x <= y <= 2 * x])  # [5, 6]: đúng các y tạo cặp mạnh với 5

[5, 6, 25, 30]
[5, 6]


### 3.4b. Tìm khoảng bằng `bisect`

`bisect_left(a, v)` trên **mảng đã sắp xếp** trả về vị trí đầu tiên có giá trị `>= v` (`O(log n)`). Dùng để nhảy thẳng tới đầu khoảng `[x, 2x]` hoặc cuối khoảng thay vì duyệt từng phần tử.

In [6]:
vals = [5, 6, 25, 30]
print(bisect_left(vals, 6))                 # 1: vị trí số đầu tiên >= 6
print(bisect_left(vals, 11))                # 2: các số < 11 là [5, 6]

1
2


### 3.5. Trie nhị phân: một "cây" các bit

**Trie nhị phân** lưu các số như **đường đi** trên cây: mỗi nút có tối đa **2 con** ứng với bit `0` và bit `1`; một số `B` bit là đường đi từ gốc xuống sâu `B` tầng, đọc các bit **từ cao xuống thấp**. Ví dụ với `B = 3` bit: `5 = 101` đi `1 -> 0 -> 1`.

Để tìm số `y` có `x ^ y` **lớn nhất** trong tập đã lưu: đi từ bit cao xuống thấp; ở mỗi bit, **ưu tiên đi nhánh có bit ngược** với bit của `x` (vì bit XOR `1` ở vị trí cao đóng góp nhiều nhất); nếu nhánh đó không tồn tại thì đi nhánh còn lại.

In [7]:
B = 3
x = 5                                       # 101
stored = [1, 2, 3]                          # các số đã lưu
best = max(x ^ y for y in stored)
print("brute:", best, "(với y =", max(stored, key=lambda y: x ^ y), ")")
# Duyệt theo bit: ưu tiên bit ngược
print("bit của x:", [(x >> b) & 1 for b in range(B - 1, -1, -1)])

brute: 7 (với y = 2 )
bit của x: [1, 0, 1]


### 3.6. Trie có bộ đếm để xóa: `count` ở mỗi nút

Ở bài này ta cần **thêm** và **xóa** số khi cửa sổ trượt. Mỗi nút giữ thêm `count` = số số đang đi qua nút đó. Xóa một số: giảm `count` dọc đường đi. Khi tìm kiếm, một nhánh chỉ "tồn tại" nếu `count > 0`.

In [8]:
class BinaryTrie:
    def __init__(self, bits: int):
        self.bits = bits
        self.child = [[-1, -1]]               # child[node] = [nút con bit 0, nút con bit 1]
        self.count = [0]                      # count[node] = số số đi qua nút

    def _walk(self, x, delta):
        node = 0
        for b in range(self.bits - 1, -1, -1):
            bit = (x >> b) & 1
            if self.child[node][bit] == -1:
                self.child[node][bit] = len(self.child)
                self.child.append([-1, -1])
                self.count.append(0)
            node = self.child[node][bit]
            self.count[node] += delta

    def add(self, x):
        self._walk(x, 1)

    def remove(self, x):
        self._walk(x, -1)

    def max_xor(self, x):
        """XOR lớn nhất của x với một số đang có trong trie (trie không rỗng)."""
        node, result = 0, 0
        for b in range(self.bits - 1, -1, -1):
            bit = (x >> b) & 1
            want = 1 - bit                    # ưu tiên bit ngược
            nxt = self.child[node][want]
            if nxt != -1 and self.count[nxt] > 0:
                result |= 1 << b
                node = nxt
            else:
                node = self.child[node][bit]
        return result


t = BinaryTrie(3)
for v in [1, 2, 3]:
    t.add(v)
print(t.max_xor(5))                 # 7 (5 ^ 2)
print(max(5 ^ y for y in [1, 2, 3]))
t.remove(2)
print(t.max_xor(5), max(5 ^ y for y in [1, 3]))

7
7
6 6


### 3.7. Hai con trỏ trên mảng đã sắp xếp

Sau khi sắp xếp `a`, với `y = a[j]` các `x` tạo cặp mạnh (`x <= y <= 2x`, tức `x >= y / 2`) là **các phần tử liên tiếp** ở bên trái, từ vị trí `left` đến `j`. Khi `y` tăng lên, ngưỡng `y / 2` tăng nên `left` **chỉ đi sang phải** (không bao giờ lùi). Đây là **cửa sổ trượt (two pointers)**: tổng số bước dịch `left` là `O(n)`.

In [9]:
a = sorted([1, 2, 3, 4, 5])
left = 0
for j, y in enumerate(a):
    while a[left] * 2 < y:                  # a[left] quá nhỏ: y > 2 * a[left]
        left += 1
    print(f"y={y}: cửa sổ = {a[left:j + 1]}")

y=1: cửa sổ = [1]
y=2: cửa sổ = [1, 2]
y=3: cửa sổ = [2, 3]
y=4: cửa sổ = [2, 3, 4]
y=5: cửa sổ = [3, 4, 5]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Với ví dụ 3, `nums = [5, 6, 25, 30]`: kiểm tra từng cặp `(x, y)` với `x <= y`:

| `(x, y)` | `y <= 2x`? | cặp mạnh? | `x ^ y` |
|---|---|---|---|
| (5, 5) | 5 <= 10 | có | 0 |
| (5, 6) | 6 <= 10 | có | 3 |
| (5, 25) | 25 <= 10 không | không | |
| (5, 30) | 30 <= 10 không | không | |
| (6, 6) | 6 <= 12 | có | 0 |
| (6, 25) | 25 <= 12 không | không | |
| (6, 30) | không | không | |
| (25, 25) | 25 <= 50 | có | 0 |
| (25, 30) | 30 <= 50 | có | **7** |
| (30, 30) | 30 <= 60 | có | 0 |

Lớn nhất là `7`. Thuật toán: **liệt kê các cặp mạnh, lấy XOR, giữ max**.

## 5. Cách 1 — Brute Force / baseline: thử mọi cặp `(i, j)`

**Ý tưởng.** Hai vòng lặp chạy mọi `i, j` (cả `i == j`); nếu `|x - y| <= min(x, y)` thì cập nhật `best = max(best, x ^ y)`. Khởi tạo `best = 0` (vì cặp `(x, x)` luôn có XOR `0`).

In [10]:
def max_xor_brute(nums: List[int]) -> int:
    best = 0                                           # cặp (x, x) luôn cho 0
    for x in nums:
        for y in nums:
            if abs(x - y) <= min(x, y):                # cặp mạnh
                best = max(best, x ^ y)
    return best


for nums, expected in examples:
    print(nums, "->", max_xor_brute(nums), "(kỳ vọng", expected, ")")

[1, 2, 3, 4, 5] -> 7 (kỳ vọng 7 )
[10, 100] -> 0 (kỳ vọng 0 )
[5, 6, 25, 30] -> 7 (kỳ vọng 7 )


In [11]:
# Liệt kê các cặp mạnh (x <= y) của ví dụ 1 cùng XOR
nums = [1, 2, 3, 4, 5]
strong = [(x, y, x ^ y) for x in nums for y in nums if x <= y and abs(x - y) <= min(x, y)]
print(len(strong), "cặp mạnh:", [(x, y) for x, y, _ in strong])
print("XOR lớn nhất:", max(v for _, _, v in strong))

11 cặp mạnh: [(1, 1), (1, 2), (2, 2), (2, 3), (2, 4), (3, 3), (3, 4), (3, 5), (4, 4), (4, 5), (5, 5)]
XOR lớn nhất: 7


**Độ phức tạp (Cách 1).** Gọi `n` là độ dài `nums`. Hai vòng lồng nhau: **Time `O(n²)`**; chỉ vài biến: **Space `O(1)`**. Với `n <= 50` (2500 cặp) rất nhanh.

**Nút thắt.** Ta xét cả những cặp `(x, y)` mà **hiển nhiên không mạnh** (`y` quá lớn so với `x`). Sau khi sắp xếp, các `y` hợp lệ cho mỗi `x` tạo thành một **khoảng liên tục** `[x, 2x]`.

## 6. Ý tưởng thuật toán / bước đột phá

**Đơn giản hóa điều kiện.** Với `x <= y`, cặp mạnh ⇔ **`y <= 2x`**. Vì vậy:

1. **Bỏ trùng và sắp xếp** các giá trị; mỗi `x` chỉ cần xét `y` trong **`[x, 2x]`** (Cách 2);
2. Để chạy tốt cả khi `n` lớn (Bài II: `n` tới `5·10⁴`, `nums[i]` tới `2²⁰`), dùng **cửa sổ trượt** trên mảng đã sắp xếp: cửa sổ chứa các `x` hợp lệ cho `y` hiện tại; và dùng **Trie nhị phân đếm số lượng** để hỏi nhanh "XOR lớn nhất của `y` với một số trong cửa sổ" trong `O(B)` (Cách 3).

## 7. Các cách tiếp theo

### Cách 2 — Bỏ trùng, sắp xếp, chỉ duyệt `y` trong `[x, 2x]`

**Giải thích.** Lấy `vals = sorted(set(nums))`. Với mỗi `i`, duyệt `j >= i` và **dừng ngay** khi `vals[j] > 2 * vals[i]` (các `y` sau còn lớn hơn). Chỉ XOR những cặp thật sự mạnh. Cách này không cần kiểm tra điều kiện trị tuyệt đối.

In [12]:
def max_xor_sorted(nums: List[int]) -> int:
    vals = sorted(set(nums))
    best = 0
    for i, x in enumerate(vals):
        for j in range(i, len(vals)):
            y = vals[j]
            if y > 2 * x:                              # mọi y sau còn lớn hơn: dừng
                break
            best = max(best, x ^ y)
    return best


for nums, expected in examples:
    print(nums, "->", max_xor_sorted(nums), "(kỳ vọng", expected, ")")

[1, 2, 3, 4, 5] -> 7 (kỳ vọng 7 )
[10, 100] -> 0 (kỳ vọng 0 )
[5, 6, 25, 30] -> 7 (kỳ vọng 7 )


In [13]:
# Code nộp LeetCode cho Cách 2
from typing import List

class Solution:
    def maximumStrongPairXor(self, nums: List[int]) -> int:
        vals = sorted(set(nums))
        best = 0
        for i, x in enumerate(vals):
            for j in range(i, len(vals)):
                y = vals[j]
                if y > 2 * x:
                    break
                best = max(best, x ^ y)
        return best

**Độ phức tạp Cách 2:** **Time `O(n log n + n · w)`** với `w` là số phần tử trong khoảng `[x, 2x]` (tối đa `n`, nên tệ nhất `O(n²)` nhưng thường nhỏ hơn); **Space `O(n)`**.

### Cách 3 — Duyệt `y` theo giá trị trong `[x, 2x]` (miền giá trị nhỏ)

**Giải thích.** Vì `nums[i] <= 100`, ta có thể lưu các giá trị trong `set` và với mỗi `x` thử **mọi số nguyên `y` trong `[x, min(2x, 100)]`**, XOR nếu `y` có trong `set`. Số bước tối đa `Σ x ≈ 5000`, độc lập với `n`. Chỉ hợp khi **miền giá trị nhỏ**.

In [14]:
def max_xor_value_range(nums: List[int]) -> int:
    present = set(nums)
    best = 0
    for x in present:
        for y in range(x, 2 * x + 1):                  # mọi y có thể tạo cặp mạnh với x
            if y in present:
                best = max(best, x ^ y)
    return best


for nums, expected in examples:
    print(nums, "->", max_xor_value_range(nums), "(kỳ vọng", expected, ")")

[1, 2, 3, 4, 5] -> 7 (kỳ vọng 7 )
[10, 100] -> 0 (kỳ vọng 0 )
[5, 6, 25, 30] -> 7 (kỳ vọng 7 )


**Độ phức tạp Cách 3:** **Time `O(Σ x)` ≈ `O(V²)`** với `V = max(nums)` (`V = 100`), **Space `O(n)`**. Không scale cho Bài II (`V` tới `2²⁰`).

### Cách 4 — Sắp xếp + cửa sổ trượt + Trie nhị phân có đếm

**Giải thích.** Sắp xếp `nums` (giữ cả số trùng, trie có bộ đếm xử lý được). Duyệt `y = a[j]` tăng dần: (1) thêm `y` vào trie; (2) trong khi `a[left] * 2 < y` (không còn tạo cặp mạnh với `y`) thì **xóa** `a[left]` khỏi trie và `left += 1`; (3) `best = max(best, trie.max_xor(y))`. Trie chỉ chứa các `x` với `x <= y <= 2x`, gồm cả `y` (cho cặp `(y, y)` với XOR `0`).

In [15]:
def max_xor_trie(nums: List[int]) -> int:
    a = sorted(nums)
    bits = max(a).bit_length()                         # đủ số bit cho mọi giá trị
    trie = BinaryTrie(bits)
    best = left = 0
    for y in a:
        trie.add(y)                                    # thêm y vào cửa sổ
        while a[left] * 2 < y:                         # a[left] quá nhỏ so với y
            trie.remove(a[left])
            left += 1
        best = max(best, trie.max_xor(y))              # XOR lớn nhất với các x trong cửa sổ
    return best


for nums, expected in examples:
    print(nums, "->", max_xor_trie(nums), "(kỳ vọng", expected, ")")

[1, 2, 3, 4, 5] -> 7 (kỳ vọng 7 )
[10, 100] -> 0 (kỳ vọng 0 )
[5, 6, 25, 30] -> 7 (kỳ vọng 7 )


In [16]:
# Code nộp LeetCode cho Cách 4 (Trie dạng lớp lồng trong Solution)
from typing import List

class Solution:
    def maximumStrongPairXor(self, nums: List[int]) -> int:
        a = sorted(nums)
        bits = max(a).bit_length()
        child, count = [[-1, -1]], [0]

        def update(x: int, delta: int) -> None:
            node = 0
            for b in range(bits - 1, -1, -1):
                bit = (x >> b) & 1
                if child[node][bit] == -1:
                    child[node][bit] = len(child)
                    child.append([-1, -1])
                    count.append(0)
                node = child[node][bit]
                count[node] += delta

        def query(x: int) -> int:
            node, res = 0, 0
            for b in range(bits - 1, -1, -1):
                bit = (x >> b) & 1
                nxt = child[node][1 - bit]
                if nxt != -1 and count[nxt] > 0:
                    res |= 1 << b
                    node = nxt
                else:
                    node = child[node][bit]
            return res

        best = left = 0
        for y in a:
            update(y, 1)
            while a[left] * 2 < y:
                update(a[left], -1)
                left += 1
            best = max(best, query(y))
        return best

**Độ phức tạp Cách 4:** **Time `O(n log n + n · B)`** với `B` là số bit (sắp xếp `O(n log n)`, mỗi phần tử thêm/xóa/hỏi `O(B)`), **Space `O(n · B)`** cho trie. Đây là cách cho Bài II.

> Với Bài I (`n <= 50`, `nums[i] <= 100`) cả bốn cách đều dư sức; Cách 2 là cách nộp ngắn gọn. Cách 4 đáng học vì là kỹ thuật chuẩn cho "XOR lớn nhất trong một cửa sổ".

## 8. TRACE từng bước

Ta theo dõi (a) Cách 2 với `nums = [5, 6, 25, 30]`; (b) Cách 4 (cửa sổ trượt + trie) với `nums = [1, 2, 3, 4, 5]`.

In [17]:
def trace_sorted(nums):
    vals = sorted(set(nums))
    print("vals =", vals)
    best = 0
    for i, x in enumerate(vals):
        for j in range(i, len(vals)):
            y = vals[j]
            if y > 2 * x:
                print(f"x={x}, y={y}: y > 2x = {2 * x} -> dừng")
                break
            value = x ^ y
            best = max(best, value)
            print(f"x={x}, y={y}: mạnh, x^y={value} (bin {bin(x)} ^ {bin(y)} = {bin(value)}), best={best}")
    print("=>", best)
    return best


assert trace_sorted([5, 6, 25, 30]) == 7

vals = [5, 6, 25, 30]
x=5, y=5: mạnh, x^y=0 (bin 0b101 ^ 0b101 = 0b0), best=0
x=5, y=6: mạnh, x^y=3 (bin 0b101 ^ 0b110 = 0b11), best=3
x=5, y=25: y > 2x = 10 -> dừng
x=6, y=6: mạnh, x^y=0 (bin 0b110 ^ 0b110 = 0b0), best=3
x=6, y=25: y > 2x = 12 -> dừng
x=25, y=25: mạnh, x^y=0 (bin 0b11001 ^ 0b11001 = 0b0), best=3
x=25, y=30: mạnh, x^y=7 (bin 0b11001 ^ 0b11110 = 0b111), best=7
x=30, y=30: mạnh, x^y=0 (bin 0b11110 ^ 0b11110 = 0b0), best=7
=> 7


In [18]:
def trace_trie(nums):
    a = sorted(nums)
    bits = max(a).bit_length()
    trie = BinaryTrie(bits)
    best = left = 0
    print("a =", a, "| bits =", bits)
    for j, y in enumerate(a):
        trie.add(y)
        while a[left] * 2 < y:
            print(f"  y={y}: a[left]={a[left]} và 2*{a[left]}={2 * a[left]} < {y} -> xóa {a[left]}")
            trie.remove(a[left])
            left += 1
        window = a[left:j + 1]
        value = trie.max_xor(y)
        best = max(best, value)
        print(f"y={y}: cửa sổ={window}, max XOR với y = {value}, best={best}")
        assert value == max(y ^ x for x in window)        # đối chiếu với tính tay
    print("=>", best)
    return best


assert trace_trie([1, 2, 3, 4, 5]) == 7

a = [1, 2, 3, 4, 5] | bits = 3
y=1: cửa sổ=[1], max XOR với y = 0, best=0
y=2: cửa sổ=[1, 2], max XOR với y = 3, best=3
  y=3: a[left]=1 và 2*1=2 < 3 -> xóa 1
y=3: cửa sổ=[2, 3], max XOR với y = 1, best=3
y=4: cửa sổ=[2, 3, 4], max XOR với y = 7, best=7
  y=5: a[left]=2 và 2*2=4 < 5 -> xóa 2
y=5: cửa sổ=[3, 4, 5], max XOR với y = 6, best=7
=> 7


Bảng tóm tắt (Cách 4, `nums = [1, 2, 3, 4, 5]`):

| `y` | cửa sổ `[x ≥ y/2]` | max `y ^ x` trong cửa sổ | `best` |
|---|---|---|---|
| 1 | `[1]` | 0 | 0 |
| 2 | `[1, 2]` | `2 ^ 1 = 3` | 3 |
| 3 | `[2, 3]` | `3 ^ 2 = 1` | 3 |
| 4 | `[2, 3, 4]` | `4 ^ 3 = 7` | 7 |
| 5 | `[3, 4, 5]` | `5 ^ 3 = 6` | 7 |

## 9. Code nộp LeetCode

Bản nộp dùng Cách 2 (sắp xếp, bỏ trùng, duyệt khoảng `[x, 2x]`): ngắn, dễ kiểm chứng, đúng với ràng buộc của Bài I. Đúng class `Solution`, đúng method `maximumStrongPairXor(self, nums: List[int]) -> int`.

In [19]:
from typing import List

class Solution:
    def maximumStrongPairXor(self, nums: List[int]) -> int:
        vals = sorted(set(nums))                   # bỏ trùng và sắp xếp
        best = 0                                   # cặp (x, x) luôn cho XOR 0
        for i, x in enumerate(vals):
            for j in range(i, len(vals)):
                y = vals[j]                        # y >= x
                if y > 2 * x:                      # không còn là cặp mạnh: y sau còn lớn hơn
                    break
                best = max(best, x ^ y)
        return best


sol = Solution()
for nums, expected in examples:
    print(nums, "->", sol.maximumStrongPairXor(nums))

[1, 2, 3, 4, 5] -> 7
[10, 100] -> 0
[5, 6, 25, 30] -> 7


## 10. Vì sao thuật toán đúng

- **Đơn giản hóa điều kiện:** với `x <= y`, `|x − y| = y − x` và `min(x, y) = x`, nên `|x − y| <= min(x, y)` ⇔ `y − x <= x` ⇔ `y <= 2x`.
- **Mọi cặp mạnh được xét:** mỗi cặp mạnh không thứ tự `{x, y}` với `x <= y` xuất hiện khi vòng ngoài tại `x` và vòng trong tại `y` (cả `y = x`). Vì mảng đã sắp xếp tăng dần, mọi `y` thỏa `x <= y <= 2x` nằm liên tục từ vị trí của `x` tới trước khi vượt `2x`; thoát vòng trong bằng `break` không bỏ sót `y` nào hợp lệ (các `y` sau lớn hơn).
- **Bỏ trùng không mất cặp:** điều kiện chỉ phụ thuộc giá trị, và cặp `(x, x)` luôn có sẵn (XOR `0`, bằng giá trị khởi tạo `best = 0`).
- **Cách 4:** khi xử lý `y`, cửa sổ chứa đúng các `x` đứng trước (hoặc bằng) `y` trong mảng sắp xếp với `2x >= y`: đó chính là tập `x` tạo cặp mạnh với `y` (điều kiện `x <= y` được bảo đảm do thứ tự). Trie trả đúng XOR lớn nhất vì ở mỗi bit cao ưu tiên nhánh ngược và chỉ đi qua nhánh có `count > 0`.
- **Tính chất kiểm được:** đáp án `>= 0`, `<= 2^B − 1`, và khi chỉ có một giá trị khác nhau thì bằng `0`.

## 11. Phân tích Time / Space Complexity

Gọi `n` là độ dài `nums`, `V` là giá trị lớn nhất, `B` là số bit (`B = ⌈log₂(V+1)⌉`).

- **Cách 1:** Time `O(n²)`, Space `O(1)`.
- **Cách 2:** Time `O(n log n + n · w)` (tệ nhất `O(n²)`), Space `O(n)`.
- **Cách 3:** Time `O(V²)` (`V = 100`), Space `O(n)`.
- **Cách 4:** Time `O(n log n + n · B)`, Space `O(n · B)`.
- Với Bài II (`n <= 5 · 10⁴`, `V < 2²⁰`): Cách 1, 2 (tệ nhất), 3 đều không đủ; Cách 4 là lời giải chuẩn.

## 12. Bộ test

Ta gom các cách vào từ điển `impls` rồi kiểm tra bằng `assert`.

In [20]:
sol = Solution()
impls = {
    "brute": max_xor_brute,
    "sorted": max_xor_sorted,
    "value_range": max_xor_value_range,
    "trie": max_xor_trie,
    "submit": sol.maximumStrongPairXor,
}

for nums, expected in examples:
    for key, fn in impls.items():
        assert fn(list(nums)) == expected, (key, nums, fn(list(nums)), expected)
print("PASS ví dụ của đề (mọi cách)")

PASS ví dụ của đề (mọi cách)


### 12.1. Các trường hợp biên

In [21]:
def check_all(nums, expected, label):
    for key, fn in impls.items():
        got = fn(list(nums))
        assert got == expected, (label, key, got, expected)
    print("PASS", label, "->", expected)


check_all([1], 0, "một phần tử: chỉ cặp (x, x)")
check_all([7, 7, 7], 0, "toàn giống nhau")
check_all([1, 2], 3, "(1, 2): 2 <= 2 là cặp mạnh, 1 ^ 2 = 3")
check_all([1, 3], 0, "(1, 3): 3 > 2 không mạnh, chỉ còn (x, x)")
check_all([4, 8], 12, "(4, 8): đúng bằng 2x là mạnh, 4 ^ 8 = 12")
check_all([4, 9], 0, "(4, 9): 9 > 8 không mạnh")
check_all([100, 99, 98], 7, "các số gần nhau đều mạnh: 100 ^ 99 = 7 là lớn nhất")
check_all([50, 100], 50 ^ 100, "(50, 100): đúng bằng 2x")
check_all([10, 100], 0, "xa nhau")

PASS một phần tử: chỉ cặp (x, x) -> 0
PASS toàn giống nhau -> 0
PASS (1, 2): 2 <= 2 là cặp mạnh, 1 ^ 2 = 3 -> 3
PASS (1, 3): 3 > 2 không mạnh, chỉ còn (x, x) -> 0
PASS (4, 8): đúng bằng 2x là mạnh, 4 ^ 8 = 12 -> 12
PASS (4, 9): 9 > 8 không mạnh -> 0
PASS các số gần nhau đều mạnh: 100 ^ 99 = 7 là lớn nhất -> 7
PASS (50, 100): đúng bằng 2x -> 86
PASS xa nhau -> 0


### 12.2. Tính chất và phản ví dụ

In [22]:
# Tính chất 1: đáp án >= 0 và < 2^B; trùng lặp không đổi đáp án
random.seed(21)
for _ in range(300):
    n = random.randint(1, 20)
    nums = [random.randint(1, 100) for _ in range(n)]
    r = sol.maximumStrongPairXor(nums)
    assert 0 <= r < 128
    assert r == sol.maximumStrongPairXor(nums + nums)           # nhân đôi mảng không đổi kết quả
    assert r == sol.maximumStrongPairXor(list(reversed(nums)))  # thứ tự không quan trọng
print("PASS đáp án trong [0, 128), bất biến với trùng lặp và thứ tự")

# Tính chất 2: điều kiện đơn giản hóa y <= 2x tương đương điều kiện gốc
for x in range(1, 60):
    for y in range(x, 130):
        assert (abs(x - y) <= min(x, y)) == (y <= 2 * x)
print("PASS |x - y| <= min(x, y)  <=>  y <= 2x khi x <= y")

# Đầu vào không bị sửa
nums = [5, 6, 25, 30]
copy = list(nums)
for key, fn in impls.items():
    fn(nums)
assert nums == copy
print("PASS đầu vào không bị sửa")

# Phản ví dụ 1: không giới hạn cặp mạnh (lấy XOR lớn nhất của mọi cặp)
def sai_khong_dieu_kien(nums):
    return max(x ^ y for x in nums for y in nums)

assert sai_khong_dieu_kien([10, 100]) == 110 and sol.maximumStrongPairXor([10, 100]) == 0
print("PASS phản ví dụ: bỏ điều kiện cặp mạnh cho", sai_khong_dieu_kien([10, 100]), "thay vì 0")

# Phản ví dụ 2: điều kiện < min thay vì <= min
def sai_nho_hon(nums):
    best = 0
    for x in nums:
        for y in nums:
            if abs(x - y) < min(x, y):
                best = max(best, x ^ y)
    return best

assert sai_nho_hon([4, 8]) == 0 and sol.maximumStrongPairXor([4, 8]) == 12
print("PASS phản ví dụ: < min bỏ sót (4, 8), cho", sai_nho_hon([4, 8]), "thay vì 12")

PASS đáp án trong [0, 128), bất biến với trùng lặp và thứ tự
PASS |x - y| <= min(x, y)  <=>  y <= 2x khi x <= y
PASS đầu vào không bị sửa
PASS phản ví dụ: bỏ điều kiện cặp mạnh cho 110 thay vì 0
PASS phản ví dụ: < min bỏ sót (4, 8), cho 0 thay vì 12


### 12.3. Đối chiếu ngẫu nhiên (có seed)

In [23]:
random.seed(2932)
for trial in range(2000):
    n = random.randint(1, 50)
    hi = random.choice([5, 20, 100])
    nums = [random.randint(1, hi) for _ in range(n)]
    ref = max_xor_brute(list(nums))
    for key, fn in impls.items():
        assert fn(list(nums)) == ref, (key, nums, fn(list(nums)), ref)
print("PASS 2000 mảng ngẫu nhiên (n tới 50) - mọi cách khớp nhau")

# Giá trị lớn (như Bài II): đối chiếu Cách 1, 2 và 4 (không dùng Cách 3 vì miền giá trị lớn)
for trial in range(100):
    n = random.randint(1, 60)
    nums = [random.randint(1, 2**20 - 1) for _ in range(n)]
    ref = max_xor_brute(list(nums))
    assert max_xor_sorted(list(nums)) == ref and max_xor_trie(list(nums)) == ref, nums
print("PASS 100 mảng giá trị lớn (tới 2^20): các cách brute, sorted, trie khớp nhau")

PASS 2000 mảng ngẫu nhiên (n tới 50) - mọi cách khớp nhau
PASS 100 mảng giá trị lớn (tới 2^20): các cách brute, sorted, trie khớp nhau


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Quên điều kiện cặp mạnh

In [24]:
def sai_khong_dieu_kien(nums):
    return max(x ^ y for x in nums for y in nums)


print("sai  :", sai_khong_dieu_kien([10, 100]), "| đúng:", Solution().maximumStrongPairXor([10, 100]))
print("sai  :", sai_khong_dieu_kien([5, 6, 25, 30]), "| đúng:", Solution().maximumStrongPairXor([5, 6, 25, 30]))

sai  : 110 | đúng: 0
sai  : 31 | đúng: 7


**Vì sao sai:** đề chỉ cho phép các cặp **mạnh** `|x − y| <= min(x, y)`. `10 ^ 100 = 110` nhưng `(10, 100)` không phải cặp mạnh, nên không được tính.

### Lỗi 2: Dùng `<` thay vì `<=` trong điều kiện

In [25]:
def sai_nho_hon(nums):
    best = 0
    for x in nums:
        for y in nums:
            if abs(x - y) < min(x, y):                 # nên là <=
                best = max(best, x ^ y)
    return best


print("sai  :", sai_nho_hon([4, 8]), "| đúng:", Solution().maximumStrongPairXor([4, 8]))
print("sai  :", sai_nho_hon([1, 2]), "| đúng:", Solution().maximumStrongPairXor([1, 2]))

sai  : 0 | đúng: 12
sai  : 0 | đúng: 3


**Vì sao sai:** điều kiện của đề là **`<=`**: `|4 − 8| = 4 <= min = 4` đúng (số lớn đúng bằng gấp đôi số nhỏ vẫn mạnh). Dùng `<` loại mất những cặp "sát biên".

### Lỗi 3: Dùng `y <= 2x` mà không chắc `x <= y`

In [26]:
def sai_khong_sap_xep(nums):
    best = 0
    for x in nums:
        for y in nums:
            if y <= 2 * x:                              # chỉ đúng khi x <= y; thiếu điều kiện cho y < x
                best = max(best, x ^ y)
    return best


print("sai  :", sai_khong_sap_xep([10, 100]), "| đúng:", Solution().maximumStrongPairXor([10, 100]))

sai  : 110 | đúng: 0


**Vì sao sai:** `y <= 2x` chỉ là dạng rút gọn khi `x <= y`. Nếu `y < x` thì điều kiện này luôn đúng (vì `y < x <= 2x`) dù cặp **không** mạnh (ví dụ `x = 100`, `y = 10`). Phải đảm bảo `x <= y` (sắp xếp hoặc hoán đổi) hoặc dùng `abs(x - y) <= min(x, y)`.

### Lỗi 4: Nhầm `^` với lũy thừa

In [27]:
print("3 ^ 4  (XOR) =", 3 ^ 4)         # 7
print("3 ** 4 (mũ)  =", 3 ** 4)        # 81
print("đáp án cho [3, 4] là", Solution().maximumStrongPairXor([3, 4]))

3 ^ 4  (XOR) = 7
3 ** 4 (mũ)  = 81
đáp án cho [3, 4] là 7


**Vì sao dễ nhầm:** trong Python, **`^` là XOR**, còn lũy thừa là `**`. Nhiều người quen toán học đọc `^` là "mũ". Hãy luôn kiểm tra bằng ví dụ nhỏ: `3 ^ 4 = 7`.

### Lỗi 5: Khởi tạo `best` âm hoặc quên cặp `(x, x)`

In [28]:
def sai_best_am(nums):
    best = -1                                           # sai: mặc định -1
    vals = sorted(set(nums))
    for i, x in enumerate(vals):
        for j in range(i + 1, len(vals)):               # sai: bỏ cặp (x, x) (j bắt đầu từ i + 1)
            y = vals[j]
            if y > 2 * x:
                break
            best = max(best, x ^ y)
    return best


print("sai  :", sai_best_am([10, 100]), "| đúng:", Solution().maximumStrongPairXor([10, 100]))
print("sai  :", sai_best_am([7]), "| đúng:", Solution().maximumStrongPairXor([7]))

sai  : -1 | đúng: 0
sai  : -1 | đúng: 0


**Vì sao sai:** đề cho phép chọn **cùng một số hai lần**, nên cặp `(x, x)` (XOR `0`) luôn hợp lệ và đáp án luôn `>= 0`. Nếu bỏ cặp này và khởi tạo `best = -1`, các mảng không có cặp mạnh khác trả `-1` thay vì `0`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Thấy "cặp có XOR lớn nhất" → nghĩ tới **Trie nhị phân** (đi ngược bit) hoặc **brute force `O(n²)`** nếu `n` nhỏ.
- Thấy điều kiện có `|x − y|` và `min` → **đơn giản hóa** khi `x <= y`: `y <= 2x` (hoặc tương tự), rồi **sắp xếp + hai con trỏ**.
- Cặp `(x, x)` được phép → đáp án `>= 0`, khởi tạo `best = 0`.
- Trie cần **xóa** khi cửa sổ trượt: dùng bộ đếm `count` ở mỗi nút.
- Miền giá trị nhỏ (`<= 100`) cho phép các lối tắt như duyệt theo giá trị.
- Kiểm chứng Trie bằng cách so với `max(y ^ x for x in cửa_sổ)` trên dữ liệu ngẫu nhiên.

## 15. Biến thể của bài

1. **Bài II** (cùng series): `n <= 5·10⁴`, `nums[i] < 2²⁰`: dùng Cách 4.
2. XOR lớn nhất **không có điều kiện cặp mạnh** (bài kinh điển "Maximum XOR of Two Numbers in an Array"): Trie hoặc tập tiền tố.
3. Cặp mạnh với điều kiện khác: `|x − y| <= k` (cố định `k`): cửa sổ theo giá trị.
4. **Đếm** số cặp mạnh (không thứ tự) trong mảng.
5. Cặp mạnh có **AND / OR lớn nhất** thay vì XOR.
6. Trả về **chính cặp** `(x, y)` cho XOR lớn nhất.

Ta thử biến thể 4 (đếm cặp mạnh) và 6 (trả về cặp):

In [29]:
def count_strong_pairs(nums):
    """Biến thể 4: số cặp mạnh không thứ tự {x, y} (chỉ số i <= j, kể cả i == j)."""
    a = sorted(nums)
    count = 0
    for i, x in enumerate(a):
        hi = bisect_left(a, 2 * x + 1)                  # các y <= 2x nằm trước vị trí hi
        count += hi - i                                  # y từ vị trí i đến hi - 1
    return count


def best_pair(nums):
    """Biến thể 6: trả về (xor lớn nhất, x, y) với x <= y."""
    vals = sorted(set(nums))
    best = (0, vals[0], vals[0])
    for i, x in enumerate(vals):
        for y in vals[i:]:
            if y > 2 * x:
                break
            if x ^ y > best[0]:
                best = (x ^ y, x, y)
    return best


print("4. số cặp mạnh của [1, 2, 3, 4, 5]:", count_strong_pairs([1, 2, 3, 4, 5]))     # 11 (đúng như đề)
print("6. cặp tốt nhất của [5, 6, 25, 30]:", best_pair([5, 6, 25, 30]))               # (7, 25, 30)
assert count_strong_pairs([1, 2, 3, 4, 5]) == 11

4. số cặp mạnh của [1, 2, 3, 4, 5]: 11
6. cặp tốt nhất của [5, 6, 25, 30]: (7, 25, 30)


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thử mọi cặp | `abs(x-y) <= min(x,y)` | `O(n²)` | `O(1)` | hiểu đề; đối chiếu |
| 2. Sắp xếp, bỏ trùng, duyệt `[x, 2x]` | `break` khi `y > 2x` | `O(n log n + n·w)` | `O(n)` | **cách nộp Bài I**, ngắn gọn |
| 3. Duyệt `y` trong `[x, 2x]` theo giá trị | miền giá trị nhỏ | `O(V²)` | `O(n)` | miền `<= 100` |
| 4. Sắp xếp + cửa sổ trượt + Trie đếm | hỏi XOR lớn nhất của `y` với cửa sổ | `O(n log n + n·B)` | `O(n·B)` | **Bài II**, `n` lớn, giá trị lớn |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `6 ^ 3`, `7 ^ 7`, `abs(4 - 9) <= min(4, 9)`, `Solution().maximumStrongPairXor([8, 4])`, `Solution().maximumStrongPairXor([9, 4])`.

**Bài 2 (tính tay).** Với `nums = [2, 3, 7, 9]`: liệt kê các cặp mạnh `(x, y)` với `x <= y`, tính XOR của từng cặp và tìm lớn nhất.

**Bài 3 (điền TODO).** Hoàn thành `max_xor_todo(nums)` theo Cách 2.

In [30]:
def max_xor_todo(nums):
    vals = sorted(set(nums))
    best = 0
    for i, x in enumerate(vals):
        for j in range(i, len(vals)):
            y = vals[j]
            # TODO 1: nếu y > 2 * x thì break
            # TODO 2: best = max(best, x ^ y)
            pass
    return best

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (các TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `max_xor_brute2(nums)` bằng hai vòng lặp và điều kiện `abs(x - y) <= min(x, y)` (Cách 1) mà không nhìn code mẫu. Kiểm tra với ba ví dụ của đề.

**Bài 5 (biến thể / mini challenge).** (a) Viết `max_xor_no_condition(nums)` trả XOR lớn nhất của **mọi** cặp (bỏ điều kiện cặp mạnh) bằng Trie nhị phân (không có cửa sổ, không xóa). (b) Viết `max_strong_pair_k(nums, k)` với điều kiện `|x − y| <= k` thay cho điều kiện gốc.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `5`, `0`, `False` (`5 > 4`), `12`, `0`. Bài 2: chỉ xét `y <= 2x`.
- **Mức 2 (vừa):** Bài 2: các cặp mạnh: `(2,2)`, `(2,3)`, `(3,3)`, `(7,7)`, `(7,9)`, `(9,9)`; XOR: `0, 1, 0, 0, 14, 0`; lớn nhất `7 ^ 9 = 14`. Bài 3: hai TODO trong comment. Bài 4: `for x in nums: for y in nums: if abs(x - y) <= min(x, y): best = max(best, x ^ y)`.
- **Mức 3 (sâu):** Bài 5(a): thêm mọi số vào trie rồi với mỗi `x` gọi `max_xor`. Bài 5(b): sắp xếp rồi cửa sổ theo `a[left] >= y - k` (trie đếm) hoặc brute force `O(n²)` với điều kiện mới.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [31]:
# Bài 1
assert 6 ^ 3 == 5 and 7 ^ 7 == 0
assert (abs(4 - 9) <= min(4, 9)) is False
assert Solution().maximumStrongPairXor([8, 4]) == 12
assert Solution().maximumStrongPairXor([9, 4]) == 0

# Bài 2
assert Solution().maximumStrongPairXor([2, 3, 7, 9]) == 7 ^ 9 == 14

# Bài 3
def max_xor_todo_ans(nums):
    vals = sorted(set(nums))
    best = 0
    for i, x in enumerate(vals):
        for j in range(i, len(vals)):
            y = vals[j]
            if y > 2 * x:
                break
            best = max(best, x ^ y)
    return best

for nums, expected in examples:
    assert max_xor_todo_ans(nums) == expected

# Bài 4
def max_xor_brute2(nums):
    best = 0
    for x in nums:
        for y in nums:
            if abs(x - y) <= min(x, y):
                best = max(best, x ^ y)
    return best

for nums, expected in examples:
    assert max_xor_brute2(nums) == expected

# Bài 5(a)
def max_xor_no_condition(nums):
    bits = max(max(nums).bit_length(), 1)
    trie = BinaryTrie(bits)
    for x in nums:
        trie.add(x)
    return max(trie.max_xor(x) for x in nums)

for _ in range(100):
    nums = [random.randint(1, 1000) for _ in range(random.randint(1, 30))]
    assert max_xor_no_condition(nums) == max(x ^ y for x in nums for y in nums)

# Bài 5(b)
def max_strong_pair_k(nums, k):
    best = 0
    for x in nums:
        for y in nums:
            if abs(x - y) <= k:
                best = max(best, x ^ y)
    return best

assert max_strong_pair_k([1, 2, 3, 4, 5], 1) == 7                     # (3, 4)
assert max_strong_pair_k([10, 100], 5) == 0
print("PASS đáp án phần 19")

PASS đáp án phần 19


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (sắp xếp, bỏ trùng, duyệt khoảng `[x, 2x]`)

```python
from typing import List

class Solution:
    def maximumStrongPairXor(self, nums: List[int]) -> int:
        vals = sorted(set(nums))
        best = 0
        for i, x in enumerate(vals):
            for j in range(i, len(vals)):
                y = vals[j]
                if y > 2 * x:
                    break
                best = max(best, x ^ y)
        return best
```

**Checklist**

- [ ] Đã đơn giản điều kiện: với `x <= y`, cặp mạnh ⇔ `y <= 2x` (dùng `<=`, không `<`)?
- [ ] Mảng đã **sắp xếp** để `x <= y` và `break` đúng?
- [ ] Cho phép cặp `(x, x)`: `j` bắt đầu từ `i`, `best` khởi tạo `0`?
- [ ] Dùng `^` cho XOR (không phải `**`)?
- [ ] Với `n`/giá trị lớn (Bài II): dùng cửa sổ trượt + Trie có bộ đếm để xóa?
- [ ] Đã thử: một phần tử, toàn giống nhau, hai số sát biên gấp đôi (`4, 8`), hai số xa nhau (`10, 100`)?
- [ ] Skeleton đúng `class Solution` / `def maximumStrongPairXor(self, nums: List[int]) -> int`?